# Explore the data lake: Bronze, Silver and Gold

Browse every table of the talent data lake as it is stored, layer by layer. Read-only: nothing here changes the lake.

| Layer | What it holds |
|---|---|
| **Bronze** | Raw copies of the sources: one Parquet snapshot per PostgreSQL table (latest ingest date), and the learning-activity JSONL files byte for byte (bad lines included) |
| **Silver** | Validated, typed tables; rejected records are in the **quarantine**, with the rule that rejected them |
| **Gold** | The ten curated business tables that answer the seven questions |

**Where it reads from** comes from the environment, like the pipeline: locally (VS Code with the `.venv` kernel) that is LocalStack's `acme-talent-lake`; in JupyterHub on EKS it is the AWS lake bucket. The first cell shows which.

In [ ]:
from pathlib import Path
import sys

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "talent_etl").is_dir())
sys.path.insert(0, str(PROJECT))

import pandas as pd
from IPython.display import display

from talent_etl import explore
from talent_etl.config import Settings
from talent_etl.io_s3 import S3Client
from talent_etl.retry import RetryPolicy

settings = Settings.from_env()
s3 = S3Client(settings.s3, RetryPolicy.from_config(settings.retry))
BUCKET = settings.s3.lake_bucket
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
print("Environment:", "local (LocalStack)" if settings.is_local else "cloud (AWS)", "| lake bucket:", BUCKET)

## Every table in every layer

In [ ]:
display(explore.overview(s3, BUCKET))

## Look at any table

Change `LAYER` and `TABLE` (names from the overview above) and run the cell. `MAX_FILES` limits how many files are read for tables split over many files (Bronze learning activity has 783 files; `None` reads all).

In [ ]:
LAYER, TABLE, MAX_FILES = "silver", "employees", None

df = explore.load(s3, BUCKET, LAYER, TABLE, max_files=MAX_FILES)
print(f"{LAYER}.{TABLE}: {len(df):,} rows x {df.shape[1]} columns")
display(df.head(20))
display(df.dtypes.rename("type").to_frame().T)

## Bronze: the sources, unchanged

A PostgreSQL snapshot keeps the database's values and types and adds lineage columns (`_run_id`, `_ingest_date`, `_source_object`, ...). The learning files are shown line by line; lines that are not valid JSON stay in `_raw`.

In [ ]:
display(explore.load(s3, BUCKET, "bronze", "employees").head(5))
learning_raw = explore.load(s3, BUCKET, "bronze", "learning_activity", max_files=5)
print(f"First 5 learning files: {len(learning_raw):,} lines")
display(learning_raw.head(5))

## Silver: validated and typed, plus the quarantine

Silver tables carry `_dq_warnings` (accepted with a warning) and the source of each row. Records that failed a rule are not in the table: they are in the quarantine, with the reason and the original record.

In [ ]:
silver_learning = explore.load(s3, BUCKET, "silver", "learning_activity", max_files=1)
display(silver_learning.head(5))

rejected = explore.quarantine(s3, BUCKET, "learning_activity")
print(f"Quarantined learning events: {len(rejected):,}")
if len(rejected):
    display(rejected["reason_codes"].astype(str).value_counts().rename("records").to_frame())
    display(rejected[["source_object", "source_line", "reason_codes", "raw_record"]].head(5))

## Gold: the curated business tables

In [ ]:
for table in explore.table_files(s3, BUCKET, "gold"):
    gold = explore.load(s3, BUCKET, "gold", table)
    print(f"gold.{table}: {len(gold):,} rows x {gold.shape[1]} columns")
display(explore.load(s3, BUCKET, "gold", "promotion_pipeline").query("is_promotion_ready").head(10))

## Which version am I looking at?

Silver and Gold are read through their pointer files, which name the published run. These are the same `_current.json` files you can open in the S3 console.

In [ ]:
print(explore.read_text(s3, BUCKET, "gold/_current.json")[:1200])